# Running model x benchmark matrices with `runners/execution.py`

This notebook explains the main pipeline for running a few benchmarks against a few models in this repository. It focuses on the objects that `execution.py` expects, the code path that runs the matrix, the type/attribute contracts of the core classes, and the shape of dataset rows consumed by benchmark families.

The runnable cells use tiny in-memory stubs so the notebook can execute without downloading Hugging Face datasets or loading large vision-language models. The final section shows the equivalent wiring for the real wrappers.

In [ ]:
from pathlib import Path
import inspect
import json
import shutil
import textwrap

repo_root = Path.cwd()
print(repo_root)

## Smallest model example: using `predict(image, prompt)`

Every benchmark ultimately needs a model object with one main method:

```python
model.predict(image, prompt) -> str
```

`image` is the visual input, `prompt` is the instruction/question, and the returned string is what the benchmark evaluates. In normal use, `model` is already an instance of a concrete model wrapper such as `GPT4`, `Qwen25VL3B`, `LlavaGemma2B`, or another `BaseModel` subclass.

In [ ]:
from PIL import Image

# Assume this already exists, for example:
# model = GPT4(model_id="gpt-4.1", max_new_tokens=16)
# model = Qwen25VL3B(max_new_tokens=16)
# model = LlavaGemma2B(max_new_tokens=16)

if False:
    image = Image.open("path/to/image.jpg").convert("RGB")
    prompt = (
        "USER: <image>\n"
        "Return exactly one short label for the main thing in this image.\n"
        "ASSISTANT:"
    )

    prediction = model.predict(image, prompt)
    prediction

The important part is the final call: `model.predict(image, prompt)`. The benchmark pipeline automates the same pattern: it extracts an image from a dataset row, builds the prompt, calls `predict(...)`, then compares the returned string against the ground truth.

## Pipeline Overview

`execution.py` does not know about individual datasets or model internals. It coordinates already-constructed run objects:

1. Build `ModelRun` objects, each with a display `name`, a model object, and optionally measured load time.
2. Build `BenchmarkRun` objects, each with a `BaseBenchmark` instance and `num_samples`.
3. Call `run_benchmark_matrix(models, benchmark_runs, output_dir=...)`.
4. For every model and benchmark pair, the runner calls `benchmark.run(model=..., n=..., label_sample_size=..., show_progress=False)`.
5. Each benchmark prepares rows, builds prompts, calls `model.predict(image, prompt)`, evaluates predictions, gathers telemetry/statistics, and returns a report dictionary.
6. `execution.py` writes one JSON file per model/benchmark pair and returns a compact list of summary dictionaries.

## The `execution.py` runner code

These are the functions that perform the matrix loop and write result files.

In [ ]:
from runners import execution

print(inspect.getsource(execution._write_report))
print(inspect.getsource(execution.run_benchmark_matrix))
print(inspect.getsource(execution.run_benchmark_runs))

## Core class contracts

The runner is deliberately duck-typed, but these are the repository contracts the objects normally implement.

| Class | Attributes and types | Key methods |
|---|---|---|
| `BaseModel` | `name: str`; concrete wrappers usually also expose `model_id: str`, `max_new_tokens: int`, and sometimes `temperature: float`, `processor`, `tokenizer`, or `model` | `predict(image: PIL.Image.Image, prompt: str) -> str`; `get_tokenizer() -> Any \| None`; `count_text_tokens(text: str) -> int \| None` |
| `ModelRun` | `name: str`; `model: object`; `load_time_seconds: float \| None = None` | `from_factory(name: str, factory: Callable[..., object], *args, **kwargs) -> ModelRun` |
| `BenchmarkRun` | `benchmark: BaseBenchmark`; `num_samples: int` | validates that `num_samples >= 1` |
| `BaseBenchmark` | `name: str`; `dataset: BaseDataset`; `MAX_PROMPT_LABELS: int = 16` | `prepare(...)`; `make_prompt(...)`; `get_image_for_row(...)`; `evaluate_prediction(...)`; `run(...)`; `build_run_statistics(...)` |
| `BaseDataset` | `name: str`; `labels: list[str]` | `__iter__()`; `get_samples(n: int) -> list[dict[str, Any]]`; `get_labels(rows) -> list[str]`; `get_labels_img(row) -> list[str]` in concrete datasets; `get_image_from_row(row) -> PIL.Image.Image`; `normalize_text(text: str) -> str` |

`execution.py` only needs each model to be passed into `benchmark.run`. `BaseBenchmark.run` is the code that actually assumes a model has `predict(image, prompt) -> str` and optionally `count_text_tokens(text)` / `max_new_tokens`.

In [ ]:
from models._base_model import BaseModel
from runners.model_run import ModelRun
from runners.benchmark_run import BenchmarkRun
from benchmarks._base_benchmark import BaseBenchmark
from dataset._base_dataset import BaseDataset

### `BaseModel`

`BaseModel` is the minimal interface expected by benchmarks. A concrete model wrapper must expose `predict(image: PIL.Image.Image, prompt: str) -> str`. Token counting is optional: `BaseBenchmark.run` checks for `count_text_tokens`, and uses `max_new_tokens` only when present to infer truncation.

| Argument / attribute | Type | Function |
|---|---|---|
| `name` | `str` | Human-readable model name used in reports and usually mirrored by `ModelRun.name`. |
| `max_new_tokens` | `int` | Optional generation budget. `execution.py` records it in summaries, and `BaseBenchmark.run` uses it to flag truncated outputs when token counting is available. |
| `model_id` | `str` | Optional external checkpoint/API model identifier, used by concrete wrappers such as `Qwen25VL3B` and `GPT4`. |
| `processor` | `Any` | Optional backend processor. `BaseModel.get_tokenizer()` checks it for `processor.tokenizer`. |
| `tokenizer` | `Any` | Optional tokenizer used by `count_text_tokens(text)`. |
| `model` | `Any` | Optional loaded backend model object used inside concrete `predict(...)` implementations. |
| `predict(image, prompt)` | `(PIL.Image.Image, str) -> str` | Required method. Produces the text prediction that each benchmark evaluates. |
| `count_text_tokens(text)` | `str -> int \| None` | Optional helper used to estimate output token counts and truncation. |

A **model wrapper** is a class that hides backend-specific model loading and inference details behind the common benchmark interface: `predict(image, prompt) -> str`. For example, `GPT4` wraps an OpenAI API call, while `Qwen25VL3B` wraps a local Hugging Face processor/model pair. The benchmark loop does not need to know which backend is underneath.

`model_id` is the backend identifier the wrapper uses to load or call a model. In `Qwen25VL3B`, it selects the Hugging Face checkpoint:

```python
# models/qwen25_vl.py
model_id: str = "Qwen/Qwen2.5-VL-3B-Instruct"
model_id = "Qwen/Qwen2.5-VL-3B-Instruct"
name = "qwen2.5-vl-3b-instruct"
return AutoProcessor.from_pretrained(self.model_id)
```

In `GPT4`, the same idea is used as the API model name:

```python
# models/gpt4.py
self.model_id = model_id
response = self.client.responses.create(
    model=self.model_id,
    ...
)
```

A **processor** is a Hugging Face object that prepares multimodal inputs for the model. It usually combines image preprocessing, text formatting/tokenization, chat template handling, tensor creation, and decoding. In `_Qwen25VLBase.predict`, inherited by `Qwen25VL3B`, the processor formats chat text, turns text plus image into tensors, and decodes generated tokens:

```python
# models/qwen25_vl.py
text = self.processor.apply_chat_template(conversation, tokenize=False, add_generation_prompt=True)
inputs = self.processor(text=[text], images=[image], padding=True, return_tensors="pt")
return self.processor.decode(generated, skip_special_tokens=True).strip()
```

A **tokenizer** converts text into token ids and can count tokens. Some wrappers expose `self.tokenizer`; others expose `self.processor.tokenizer`. `BaseModel.get_tokenizer()` supports both patterns:

```python
# models/_base_model.py
tokenizer = getattr(self, "tokenizer", None)
processor = getattr(self, "processor", None)
tokenizer = getattr(processor, "tokenizer", None)
encoded = tokenizer(text or "", add_special_tokens=False, return_attention_mask=False, return_token_type_ids=False)
```

`Llava` also uses `processor.tokenizer` directly for streaming and prompt-template handling:

```python
# models/llava.py
streamer = TextIteratorStreamer(self.processor.tokenizer, skip_prompt=True, skip_special_tokens=True)
tokenizer = getattr(self.processor, "tokenizer", None)
chat_template = getattr(tokenizer, "chat_template", "") or ""
```

In [ ]:
print(inspect.getsource(BaseModel))

  Rough split in this repo:

  - Vision-language models with a processor: Qwen2.5-VL, LLaVA, LLaVA-OneVision, Gemma/PaliGemma, Falcon VLM.
    These use AutoProcessor or a model-specific processor class because they need image preprocessing plus text tokenization.
  - Custom remote-code multimodal models: MiniCPM, InternVL.
    These often use AutoTokenizer plus custom image preprocessing/chat methods rather than a standard AutoProcessor.

### `ModelRun`

`ModelRun` is a small dataclass used by `execution.py` to pair a model object with the name that should be written to result paths and summaries.

| Argument / attribute | Type | Function |
|---|---|---|
| `name` | `str` | Model label used in summary dictionaries and output filenames like `{model}_{benchmark}.json`. |
| `model` | `object` | The constructed model object passed into `benchmark.run(...)`; normally a `BaseModel` subclass. |
| `load_time_seconds` | `float \| None` | Optional measured model construction time. If present, `execution.py` copies it into `report["stats"]["model_load_time_seconds"]`. |
| `factory` | `Callable[..., object]` | `from_factory(...)` argument used to construct a model while timing the load. |
| `*args`, `**kwargs` | `Any` | Constructor arguments forwarded by `from_factory(...)` to `factory`. |

In [ ]:
print(inspect.getsource(ModelRun))

### `BenchmarkRun`

`BenchmarkRun` is the benchmark-side equivalent of `ModelRun`. It tells the matrix runner which benchmark object to run and how many samples to request.

| Argument / attribute | Type | Function |
|---|---|---|
| `benchmark` | `BaseBenchmark` | Prepared benchmark instance, including its dataset and prompt/evaluation behavior. |
| `num_samples` | `int` | Number of dataset rows to evaluate. Must be at least `1`; passed to `benchmark.run(...)` as `n`. |
| `label_sample_size` | `int` | Not stored directly. `execution.py` computes it as `max(4, num_samples)` and passes it to `benchmark.run(...)` so benchmarks can sample labels from enough rows. |

In [ ]:
print(inspect.getsource(BenchmarkRun))

### `BaseDataset`

`BaseDataset` standardizes access to rows, images, and labels. Benchmark classes do not require a Hugging Face dataset specifically; they require an object with this interface.

| Argument / attribute | Type | Function |
|---|---|---|
| `name` | `str` | Dataset name included in benchmark reports. |
| `labels` | `list[str]` | Candidate label cache used by classification and detection-style prompts. |
| `__iter__()` | `() -> Iterable[dict[str, Any]]` | Provides row iteration for concrete datasets. |
| `get_samples(n)` | `int -> list[dict[str, Any]]` | Returns the first or sampled `n` rows used by `BaseBenchmark.prepare(...)`. |
| `get_labels(rows)` | `Iterable[dict[str, Any]] -> list[str]` | Builds candidate labels from sampled rows. |
| `get_labels_img(row)` | `dict[str, Any] -> list[str]` | Common concrete helper returning valid labels for a single row. |
| `get_image_from_row(row)` | `dict[str, Any] -> PIL.Image.Image` | Extracts or decodes the image passed to `model.predict(...)`. |
| `normalize_text(text)` | `str -> str` | Normalizes labels, answers, and predictions before comparison. |
| `get_question_from_row(row)` | `dict[str, Any] -> str` | Optional VQA/multiple-choice helper for question extraction. |
| `get_answers_from_row(row)` | `dict[str, Any] -> list[str]` | Optional VQA helper for accepted answers. |
| `get_captions_from_row(row)` | `dict[str, Any] -> list[str]` | Optional captioning helper for reference captions. |
| `get_annotations_for_row(row)` | `dict[str, Any] -> list[dict[str, Any]]` | Optional detection helper for labeled bounding boxes. |

In [ ]:
print(inspect.getsource(BaseDataset))

### `BaseBenchmark`

`BaseBenchmark` owns the per-sample evaluation loop. It prepares rows and labels, builds prompts, obtains an image, calls the model, evaluates the prediction, records boxes/labels/statistics, and returns the report consumed by `execution.py`.

| Argument / attribute | Type | Function |
|---|---|---|
| `dataset` | `BaseDataset` | Source of rows, images, labels, answers, captions, or annotations. Required by `__init__`. |
| `name` | `str` | Benchmark name used in reports, summaries, and output filenames. Required by `__init__`. |
| `MAX_PROMPT_LABELS` | `int` | Caps how many candidate labels are shown in prompts; default is `16`, detection uses `10`. |
| `model` | `BaseModel` | Argument to `run(...)`; receives each image/prompt pair through `model.predict(...)`. |
| `n` | `int` | Argument to `run(...)` and `prepare(...)`; number of rows to evaluate. |
| `label_sample_size` | `int` | Argument to `run(...)` and `prepare(...)`; number of rows used to build candidate labels. |
| `show_progress` | `bool` | Argument to `run(...)`; controls progress printing. `execution.py` sets it to `False`. |
| `on_sample` | `Callable[[dict[str, Any]], None] \| None` | Optional callback called with each full sample payload before it is compacted into the final report. |
| `row` | `dict[str, Any]` | Dataset row passed through prompting, image extraction, and evaluation helpers. |
| `labels` | `list[str]` | Candidate labels from `get_candidate_labels(...)`; often rendered in classification/detection prompts. |
| `image` | `Any \| None` | Image or visual input passed to `model.predict(...)` and evaluation helpers. |
| `prediction` | `str` | Model output text that benchmark-specific evaluation parses and scores. |
| `prepare(n, label_sample_size)` | `(int, int) -> tuple[list[dict[str, Any]], list[str]]` | Selects rows and candidate labels for a run. |
| `make_prompt(labels, row, image)` | `(list[str], dict[str, Any] \| None, Any \| None) -> str` | Creates the prompt sent to the model. |
| `evaluate_prediction(row, prediction, image)` | `(dict[str, Any], str, Any \| None) -> tuple[bool, list[str], dict[str, Any]]` | Scores a prediction and returns correctness, valid labels, and extra metrics. |
| `analyze_prediction(...)` | `(...) -> dict[str, Any]` | Computes behavior counts such as hallucinated labels, false positives, false negatives, and detection count. |

In [ ]:
print(inspect.getsource(BaseBenchmark.__init__))
print(inspect.getsource(BaseBenchmark.prepare))
print(inspect.getsource(BaseBenchmark.make_prompt))
print(inspect.getsource(BaseBenchmark.evaluate_prediction))

## The benchmark execution path

The most important method after `execution.py` is `BaseBenchmark.run`. It is shared by classification, VQA, captioning, multiple-choice, detection, and video benchmark subclasses. Subclasses mostly specialize prompt construction, image preparation, label extraction, and evaluation.

In [ ]:
print(inspect.getsource(BaseBenchmark.prepare))
print(inspect.getsource(BaseBenchmark.run))
print(inspect.getsource(BaseBenchmark.build_run_statistics))

## Family-specific benchmark behavior

Concrete benchmark classes are thin wrappers around family base classes. For example, `ImageNet1kBenchmark` constructs an `ImageNet1k` dataset and then delegates to `ClassificationBenchmark`; `GQABenchmark` delegates to `VisualQABenchmark`; `OpenImagesV4DetectionBenchmark` delegates to `DetectionBenchmark`.

The cells below print the family logic that changes prompts and evaluation.

In [ ]:
from benchmarks.classification._classification import ClassificationBenchmark
from benchmarks.visual_qa._visual_qa import VisualQABenchmark
from benchmarks.captioning._captioning import CaptioningBenchmark
from benchmarks.multiple_choice._multiple_choice import MultipleChoiceBenchmark
from benchmarks.detection._detection import DetectionBenchmark

for cls in [ClassificationBenchmark, VisualQABenchmark, CaptioningBenchmark, MultipleChoiceBenchmark, DetectionBenchmark]:
    print(f"\n# {cls.__name__}")
    print(inspect.getsource(cls))

## Example dataset row shapes

Rows are plain dictionaries. The exact keys vary by dataset and benchmark family, but the benchmark base classes look for a small set of common fields.

Classification row:

```python
{
    "id": "img-0001",
    "image": PIL.Image.Image(...),
    "label": "tabby cat",     # some HF datasets use an integer class id instead
}
```

Visual QA row:

```python
{
    "id": "gqa-42",
    "image_id": "2407890",
    "image": PIL.Image.Image(...),
    "question": "What color is the bus?",
    "answers": ["yellow"],
}
```

Captioning row:

```python
{
    "id": "flickr-17",
    "image": PIL.Image.Image(...),
    "captions": [
        "A child in a red shirt is jumping over a puddle.",
        "A young child leaps outside near water.",
    ],
}
```

Multiple-choice row:

```python
{
    "id": "cc-9",
    "image": PIL.Image.Image(...),
    "question": "Which caption matches the image?",
    "choices": ["A dog in a field", "A train at a station", "A bowl of soup"],
    "answer": "A dog in a field",
}
```

Detection row:

```python
{
    "id": "oid-3",
    "image": PIL.Image.Image(...),
    "objects": [
        {"label": "car", "xmin": 0.10, "ymin": 0.20, "xmax": 0.55, "ymax": 0.60},
        {"label": "person", "xmin": 0.62, "ymin": 0.18, "xmax": 0.78, "ymax": 0.72},
    ],
}
```

Video row:

```python
{
    "video_id": "ucf-12",
    "frames": [PIL.Image.Image(...), PIL.Image.Image(...), PIL.Image.Image(...)],
    "label_text": "playing guitar",
}
```

## Runnable tiny matrix

This example runs two toy models against two toy benchmarks. It exercises the same `run_benchmark_matrix` API used by real runs, but avoids external data/model downloads.

In [ ]:
from typing import Any, Dict, Iterable, List
from PIL import Image

from runners.execution import run_benchmark_matrix
from runners.model_run import ModelRun
from runners.benchmark_run import BenchmarkRun


def square(color: str, size: int = 32) -> Image.Image:
    return Image.new("RGB", (size, size), color)


class TinyClassificationDataset(BaseDataset):
    name = "tiny_classification"

    def __init__(self):
        self.rows = [
            {"id": "cls-1", "image": square("red"), "label": "red square"},
            {"id": "cls-2", "image": square("blue"), "label": "blue square"},
        ]
        self.labels = ["red square", "blue square"]

    def __iter__(self) -> Iterable[Dict[str, Any]]:
        return iter(self.rows)

    def get_samples(self, n: int) -> List[Dict[str, Any]]:
        return self.rows[:n]

    def get_labels(self, rows) -> List[str]:
        return list(self.labels)

    def get_labels_img(self, row) -> List[str]:
        return [row["label"]]

    def get_image_from_row(self, row: Dict[str, Any]) -> Image.Image:
        return row["image"]


class TinyQADataset(BaseDataset):
    name = "tiny_qa"

    def __init__(self):
        self.rows = [
            {
                "id": "qa-1",
                "image": square("green"),
                "question": "What color is the square?",
                "answers": ["green"],
            },
            {
                "id": "qa-2",
                "image": square("yellow"),
                "question": "What color is the square?",
                "answers": ["yellow"],
            },
        ]
        self.labels = []

    def __iter__(self) -> Iterable[Dict[str, Any]]:
        return iter(self.rows)

    def get_samples(self, n: int) -> List[Dict[str, Any]]:
        return self.rows[:n]

    def get_labels(self, rows) -> List[str]:
        return []

    def get_labels_img(self, row) -> List[str]:
        return list(row.get("answers", []))

    def get_image_from_row(self, row: Dict[str, Any]) -> Image.Image:
        return row["image"]


class PromptAwareToyModel(BaseModel):
    def __init__(self, name: str, max_new_tokens: int = 16):
        self.name = name
        self.max_new_tokens = max_new_tokens

    def predict(self, image: Image.Image, prompt: str) -> str:
        # Deliberately simple: use the image's top-left pixel and the prompt style.
        color = image.getpixel((0, 0))
        if "Question:" in prompt:
            if color == (0, 128, 0):
                return "green"
            if color == (255, 255, 0):
                return "yellow"
        if color == (255, 0, 0):
            return "red square"
        if color == (0, 0, 255):
            return "blue square"
        return "unknown"


models = [
    ModelRun(name="toy-a", model=PromptAwareToyModel("toy-a", max_new_tokens=16), load_time_seconds=0.0),
    ModelRun(name="toy-b", model=PromptAwareToyModel("toy-b", max_new_tokens=16), load_time_seconds=0.0),
]

benchmarks = [
    BenchmarkRun(benchmark=ClassificationBenchmark(TinyClassificationDataset(), name="tiny_classification"), num_samples=2),
    BenchmarkRun(benchmark=VisualQABenchmark(TinyQADataset(), name="tiny_qa"), num_samples=2),
]

output_dir = repo_root / ".tmp" / "execution_pipeline_notebook_results"
shutil.rmtree(output_dir, ignore_errors=True)

summaries = run_benchmark_matrix(models=models, benchmark_runs=benchmarks, output_dir=output_dir)
summaries

In [ ]:
first_result_path = Path(summaries[0]["results_path"])
payload = json.loads(first_result_path.read_text(encoding="utf-8"))
print(json.dumps(payload, indent=2, ensure_ascii=False, default=str)[:4000])

## Wiring real models and benchmarks

The real run looks the same. The difference is that model constructors may download or load large checkpoints, and dataset constructors may stream/download Hugging Face datasets. Keep `num_samples` small while smoke testing.

`ModelRun.from_factory(...)` is useful when you want load time recorded in the report stats.

In [ ]:
if False:
    from models import LlavaGemma2B, Qwen25VL3B, GPT4
    from benchmarks import GQABenchmark, Flickr30kBenchmark, OpenImagesV4DetectionBenchmark

    real_models = [
        ModelRun.from_factory("llava-gemma-2b", LlavaGemma2B, max_new_tokens=16),
        ModelRun.from_factory("qwen2.5-vl-3b-instruct", Qwen25VL3B, max_new_tokens=16),
        # Requires OPENAI_API_KEY and the openai package.
        # ModelRun.from_factory("gpt-4.1", GPT4, model_id="gpt-4.1", max_new_tokens=16),
    ]

    real_benchmarks = [
        BenchmarkRun(benchmark=GQABenchmark(streaming=True), num_samples=8),
        BenchmarkRun(benchmark=Flickr30kBenchmark(streaming=True), num_samples=8),
        BenchmarkRun(benchmark=OpenImagesV4DetectionBenchmark(streaming=True), num_samples=8),
    ]

    real_summaries = run_benchmark_matrix(
        models=real_models,
        benchmark_runs=real_benchmarks,
        output_dir=repo_root / "results",
    )
    real_summaries

## Result file shape

Each result JSON file is written as:

```python
{
    "model": "toy-a",
    "benchmark": "tiny_classification",
    "report": {
        "benchmark": "tiny_classification",
        "dataset": "tiny_classification",
        "num_samples": 2,
        "num_candidate_labels": 2,
        "results": [
            {
                "index": 1,
                "prompt_labels": ["red square", "blue square"],
                "prediction": "red square",
                "correct": True,
                "valid_labels": ["red square"],
                "ground_truth_boxes": [],
                "predicted_boxes": [],
                "stats": {...},
            }
        ],
        "stats": {
            "wall_clock_time_seconds": 0.0,
            "success_count": 2,
            "failure_count": 0,
            "model_load_time_seconds": 0.0,
            ...
        },
    },
}
```

The summary returned by `run_benchmark_matrix` is intentionally smaller: model name, benchmark name, sample count, `max_new_tokens`, and the JSON path.